<a href="https://colab.research.google.com/github/gianluca379/labo3-2026ba/blob/main/z251_AutoARIMA_bayesiana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0.1 Init ambiente Google Colab

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para disponer de persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Bajar datasets si hace falta

In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp "/content/buckets/b1/kaggle/kaggle (2).json"  ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Modelo AutoARIMA

## 1.1 Init Experimento

In [3]:
# instalacion de paquetes que NO vienen por default en Colab
!pip install uv
!uv pip install -q kaggle
!uv pip install -q pmdarima
!uv pip install -q optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.3/26.3 MB 71.7 MB/s eta 0:00:00


In [4]:
# funcion para hacer submits a Kaggle
def kaggle_submit(competencia, archivo, mensaje):

  # comando
  comando = f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"'
  # ejecucion
  os.system(comando)


In [5]:
import os as os
import numpy as np
import polars as pl
from pmdarima import auto_arima

import warnings
warnings.filterwarnings("ignore")

Por favor, cargar aqui SU semilla primigenia

In [6]:
# defino los parametros
PARAM = {'experimento':'AA01',
  'kaggle_competition':'labo-iii-2026-ba',
  'semilla_primigenia':199410
}

In [7]:
# creo la carpeta del experimento y hago el chdir
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
print(ruta)
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

/content/buckets/b1/exp/AA01


## 1.2 Init AutoARIMA

In [8]:
# cargo el dataset del sell-in
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

In [9]:
# agrupo por product_id, periodo
tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
)

tb_ventas = tb_ventas.sort(["product_id", "periodo"])

In [10]:
# cargo la tabla "apredecir" que contiene los 780 productos que deben predecirse las ventas de 202002
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")


# Filtro tb_ventas a solo las que debo predecir
print(tb_ventas.height)
tb_ventas = tb_ventas.join(tb_apredecir,
  on="product_id",
  how="inner"
)
print(tb_ventas.height)
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

31243
22349


In [11]:
display(tb_ventas)

product_id,periodo,tn
i64,i64,f64
20001,201701,934.77222
20001,201702,798.0162
20001,201703,1303.35771
20001,201704,1069.9613
20001,201705,1502.20132
…,…,…
21276,201908,0.01265
21276,201909,0.01856
21276,201910,0.02079


Opcion de empiojar el dataset, agregando ruido relativo a las ventas
<br>Un Experimento no se le niega a nadie

In [12]:
empiojar=False
empiojar_ruido=0.3

if empiojar:
  np.random.seed(PARAM['semilla_primigenia'])
  tb_ventas = tb_ventas.sort(["product_id", "periodo"])
  # vector con el ruido multiplicativo de media 1.0  y desvio  'empiojar_ruido'
  noise_multiplier = np.random.lognormal(mean=0.0, sigma=empiojar_ruido, size=tb_ventas.height)

  tb_ventas = tb_ventas.with_columns(
    (pl.col("tn") * pl.lit(noise_multiplier)).alias("tn")
  )


In [13]:
display(tb_ventas)

product_id,periodo,tn
i64,i64,f64
20001,201701,934.77222
20001,201702,798.0162
20001,201703,1303.35771
20001,201704,1069.9613
20001,201705,1502.20132
…,…,…
21276,201908,0.01265
21276,201909,0.01856
21276,201910,0.02079


In [14]:
# donde voy a guardar el resultado final
arch_tb_final = 'tb_final.csv'

try:
    # Attempt to load the existing file
    tb_final = pl.read_csv(arch_tb_final)
except FileNotFoundError:
    # donde voy a guardar el resultado final
    tb_final = tb_apredecir.clone()
    tb_final = tb_final.with_columns(pl.lit(None).cast(pl.Float64).alias('tn'))


tb_final = tb_final.sort(["product_id"])


# cuento cuantos registros NO puedieron calcularse
# en mi corrida esto da  209

tb_final["tn"].is_null().sum()


0

In [15]:
correrdeCero=True

if correrdeCero:
  tb_final = tb_apredecir.clone()
  tb_final = tb_final.with_columns(pl.lit(None).cast(pl.Float64).alias('tn'))


tb_final = tb_final.sort(["product_id"])

In [16]:
display(tb_final)

product_id,tn
i64,f64
20001,null
20002,null
20003,null
20004,null
20005,null
…,…
21263,null
21265,null
21266,null


## 1.3 Búsqueda bayesiana de parámetros (Optuna)

En vez de usar los parámetros de `auto_arima` fijos "a mano" (`max_p=3, max_q=3, max_P=2, max_Q=2`, etc.), buscamos automáticamente la mejor combinación con Optuna, midiendo el error con **backtesting**: escondemos los últimos 2 meses de cada serie, entrenamos con el resto, predecimos esos 2 meses, y los comparamos contra el valor real (que sí tenemos, porque es pasado). Así no hace falta gastar submits de Kaggle para comparar configuraciones.

Para que la búsqueda sea rápida, no se corre sobre los 780 productos: se usa la muestra por volumen armada en `z101_EDA_01_con_seleccion_productos.ipynb` (sección 2), guardada en `productos_muestra_bayesiana.csv`. Si todavía no la generaste, corré primero ese notebook.

In [17]:
import optuna
import random
import time
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_HOLDOUT = 2   # meses que escondemos para validar (igual al horizonte real: 202001/202002)

### 1.3.1 Preparar las series y el backtest de un producto

In [18]:
def preparar_series(tb_ventas: pl.DataFrame) -> dict:
    """
    Convierte tb_ventas (product_id, periodo, tn) en un diccionario
    {product_id: np.array de tn ordenado cronologicamente}.
    """
    tb_ventas_ordenado = tb_ventas.sort(["product_id", "periodo"])
    series = {}
    productos_todos = tb_ventas_ordenado.select("product_id").unique().get_column("product_id").to_list()
    for p in productos_todos:
        vals = (
            tb_ventas_ordenado.filter(pl.col("product_id") == p)
            .get_column("tn")
            .to_numpy()
        )
        series[p] = vals
    return series


series = preparar_series(tb_ventas)
print(f"Series preparadas: {len(series)} productos")

Series preparadas: 780 productos


In [19]:
def backtest_producto(serie, params: dict, n_holdout: int = N_HOLDOUT):
    """
    Entrena con serie[:-n_holdout] y predice los ultimos n_holdout periodos.
    Devuelve (prediccion, real) o (None, None) si no se pudo.
    """
    if len(serie) <= n_holdout + 3:
        return None, None

    train = serie[:-n_holdout]
    real = serie[-n_holdout:]

    try:
        modelo = auto_arima(
            train,
            seasonal=params.get("seasonal", True),
            m=params.get("m", 12),
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            max_p=params.get("max_p", 3),
            max_q=params.get("max_q", 3),
            max_P=params.get("max_P", 2),
            max_Q=params.get("max_Q", 2),
            random=True,
            random_state=PARAM['semilla_primigenia'],
            n_fits=params.get("n_fits", 10),
        )
        pred = modelo.predict(n_periods=n_holdout)
        pred = np.clip(np.asarray(pred), 0, None)   # no tiene sentido tn negativa
        return pred, real
    except Exception:
        return None, None


def evaluar_configuracion(series: dict, productos: list, params: dict,
                           n_holdout: int = N_HOLDOUT) -> dict:
    """
    Error agregado tipo Total Forecast Error: sum(|real-pred|)/sum(real),
    medido sobre el ultimo mes del holdout (equivalente a 202002).
    OJO: confirmar contra la metrica EXACTA de tu competencia de Kaggle.
    """
    suma_abs_error, suma_real = 0.0, 0.0
    n_ok, n_fail = 0, 0

    for p in productos:
        serie = series.get(p)
        if serie is None:
            continue
        pred, real = backtest_producto(serie, params, n_holdout)
        if pred is None:
            n_fail += 1
            continue
        suma_abs_error += abs(real[-1] - pred[-1])
        suma_real += real[-1]
        n_ok += 1

    if n_ok == 0 or suma_real == 0:
        return {"forecast_error": np.nan, "n_ok": n_ok, "n_fail": n_fail}

    return {"forecast_error": suma_abs_error / suma_real, "n_ok": n_ok, "n_fail": n_fail}

### 1.3.2 Cargar la muestra de productos por volumen (armada en el EDA)

In [20]:
ruta_muestra = "/content/buckets/b1/exp/AA01/productos_muestra_bayesiana.csv"

try:
    productos_muestra = (
        pl.read_csv(ruta_muestra)
        .get_column("product_id")
        .to_list()
    )
    print(f"Muestra cargada desde el EDA: {len(productos_muestra)} productos")
except FileNotFoundError:
    # fallback: si todavia no corriste el notebook de seleccion por volumen,
    # usamos una muestra aleatoria para no frenar el flujo (pero conviene
    # correr el EDA y volver a ejecutar esta celda despues)
    print("No se encontro productos_muestra_bayesiana.csv -> usando muestra aleatoria de respaldo")
    random.seed(PARAM['semilla_primigenia'])
    todos_los_productos = list(series.keys())
    productos_muestra = random.sample(todos_los_productos, min(100, len(todos_los_productos)))

print(f"Productos a usar en la busqueda: {len(productos_muestra)}")

Muestra cargada desde el EDA: 161 productos
Productos a usar en la busqueda: 161


### 1.3.3 Estudio bayesiano con persistencia en Drive

El estudio se guarda en SQLite en tu Drive: si Colab se desconecta a mitad de la búsqueda, volvés a correr esta misma celda y Optuna sigue agregando trials en vez de arrancar de cero.

In [21]:
def correr_busqueda_bayesiana(series, productos_muestra, db_path, study_name="autoarima_tuning",
                              n_trials=15, timeout_seg=None):
    storage = f"sqlite:///{db_path}"

    def objective(trial):
        params = {
            "seasonal": trial.suggest_categorical("seasonal", [True, False]),
            "max_p": trial.suggest_int("max_p", 1, 5),
            "max_q": trial.suggest_int("max_q", 1, 5),
            "max_P": trial.suggest_int("max_P", 0, 3),
            "max_Q": trial.suggest_int("max_Q", 0, 3),
            "n_fits": trial.suggest_int("n_fits", 5, 20),
        }
        t0 = time.time()
        res = evaluar_configuracion(series, productos_muestra, params)
        trial.set_user_attr("n_ok", res["n_ok"])
        trial.set_user_attr("n_fail", res["n_fail"])
        trial.set_user_attr("tiempo_seg", round(time.time() - t0, 1))
        error = res["forecast_error"]
        return error if not np.isnan(error) else 1e6

    study = optuna.create_study(
        study_name=study_name,
        storage=storage,
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=PARAM['semilla_primigenia']),
        load_if_exists=True,   # retoma si ya existe, no arranca de cero
    )

    print(f"Trials ya corridos en estudios previos: {len(study.trials)}")
    study.optimize(objective, n_trials=n_trials, timeout=timeout_seg,
                    show_progress_bar=True, catch=(Exception,))

    print("\n=== Mejor configuracion hasta ahora ===")
    print(study.best_params)
    print(f"Error: {study.best_value:.4f}")
    return study


db_path = ruta + "/optuna_study.db"

study = correr_busqueda_bayesiana(
    series=series,
    productos_muestra=productos_muestra,
    db_path=db_path,
    n_trials=15,        # empezar chico; volver a correr esta celda para sumar mas trials
    timeout_seg=1800,   # corta a los 30 min pase lo que pase
)

Trials ya corridos en estudios previos: 0


  0%|          | 0/15 [00:00<?, ?it/s]


=== Mejor configuracion hasta ahora ===
{'seasonal': True, 'max_p': 5, 'max_q': 5, 'max_P': 1, 'max_Q': 2, 'n_fits': 17}
Error: 0.2616


In [22]:
df_trials = study.trials_dataframe()
display(df_trials.sort_values("value").head(10))

,number,value,datetime_start,datetime_complete,duration,params_max_P,params_max_Q,params_max_p,params_max_q,params_n_fits,params_seasonal,user_attrs_n_fail,user_attrs_n_ok,user_attrs_tiempo_seg,state
0,0,0.261575,2026-07-01 16:14:43.327343,2026-07-01 16:24:21.744620,0 days 00:09:38.417277,1,2,5,5,17,True,16,145,578.2,COMPLETE
14,14,0.265905,2026-07-01 16:43:15.887272,2026-07-01 16:53:58.056863,0 days 00:10:42.169591,2,3,2,4,14,True,16,145,642.0,COMPLETE
11,11,0.293171,2026-07-01 16:31:53.255076,2026-07-01 16:35:37.011734,0 days 00:03:43.756658,2,3,3,4,20,False,2,159,223.6,COMPLETE
4,4,0.293171,2026-07-01 16:24:22.490430,2026-07-01 16:28:05.160354,0 days 00:03:42.669924,3,3,3,4,17,False,2,159,222.5,COMPLETE
12,12,0.293171,2026-07-01 16:35:37.048511,2026-07-01 16:39:19.446835,0 days 00:03:42.398324,1,2,3,4,17,False,2,159,222.2,COMPLETE
9,9,0.293947,2026-07-01 16:28:06.239430,2026-07-01 16:31:52.953356,0 days 00:03:46.713926,3,1,4,2,7,False,2,159,226.5,COMPLETE
13,13,0.293960,2026-07-01 16:39:19.484387,2026-07-01 16:43:15.844969,0 days 00:03:56.360582,3,2,4,5,18,False,2,159,236.2,COMPLETE
2,2,1000000.000000,2026-07-01 16:24:22.025077,2026-07-01 16:24:22.217431,0 days 00:00:00.192354,0,2,3,1,14,True,161,0,0.0,COMPLETE
7,7,1000000.000000,2026-07-01 16:28:05.671129,2026-07-01 16:28:05.904011,0 days 00:00:00.232882,1,0,1,5,16,True,161,0,0.0,COMPLETE
6,6,1000000.000000,2026-07-01 16:28:05.435194,2026-07-01 16:28:05.636543,0 days 00:00:00.201349,1,0,1,3,20,True,161,0,0.0,COMPLETE


### 1.3.4 Guardar la mejor configuración para usarla en las pasadas siguientes

In [23]:
mejor_params = study.best_params
print("Parametros que se van a usar en la Primera y Segunda pasada:")
print(mejor_params)

Parametros que se van a usar en la Primera y Segunda pasada:
{'seasonal': True, 'max_p': 5, 'max_q': 5, 'max_P': 1, 'max_Q': 2, 'n_fits': 17}


## 1.4 Primera pasada AutoARIMA

Igual que la primera pasada original, pero usando `mejor_params` (salido de la búsqueda bayesiana) en vez de valores fijos a mano, y corriendo sobre **todos** los productos (no la muestra).

In [24]:
# solo los productos que faltan

productos = tb_final.filter(
    pl.col("tn").is_null()
).select("product_id" ).get_column("product_id").to_list()


# Es fundamental que tb_ventas este ORDENADO
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

for producto in productos:

  print(producto, end=' ')
  tn_values = tb_ventas.filter(pl.col("product_id") == producto).select(["tn"])

  try:
    modelo = auto_arima(
      tn_values,
      seasonal=mejor_params.get("seasonal", True),
      m=12, # estacinalidad cada 12
      stepwise=True,
      suppress_warnings=True,
      error_action="ignore",
      max_p=mejor_params.get("max_p", 3), max_q=mejor_params.get("max_q", 3),
      max_P=mejor_params.get("max_P", 2), max_Q=mejor_params.get("max_Q", 2),
      random=True,
      random_state=PARAM['semilla_primigenia'],
      n_fits=mejor_params.get("n_fits", 10) # cantidad de iteraciones de busqueda AUTO
    )

    # predigo el periodo+1 y periodo+2
    forecast = modelo.predict(n_periods=2)
    mesmasdos = forecast[1]  # el segundo elemento del vector

    tb_final = tb_final.with_columns(
       pl.when(pl.col("product_id") == producto)
      .then(mesmasdos if mesmasdos >=0  else 0 )
      .otherwise(pl.col("tn")).alias("tn")
    )

  except Exception as e:
    print(f"  ERROR for  {producto} ")

20001 20002 20003 20004 20005 20006 20007 20008 20009 20010 20011 20012 20013 20014 20015 20016 20017 20018 20019 20020 20021 20022 20023 20024 20025 20026 20027 20028 20029 20030 20031 20032   ERROR for  20032 
20033 20035 20037 20038 20039 20041 20042 20043 20044 20045 20046 20047 20049 20050 20051 20052 20053 20054 20055 20056 20057 20058 20059 20061 20062 20063 20065 20066 20067 20068 20069 20070 20071 20072 20073 20074 20075 20076 20077 20079 20080 20081 20082 20084 20085   ERROR for  20085 
20086 20087 20089   ERROR for  20089 
20090 20091 20092 20093 20094 20095 20096 20097 20099 20100 20101 20102 20103 20106 20107 20108 20109 20111 20112 20114 20116 20117 20118 20119 20120 20121 20122 20123 20124 20125 20126 20127   ERROR for  20127 
20129 20130 20132 20133 20134 20135   ERROR for  20135 
20137 20138 20139 20140 20142 20143 20144 20145 20146 20148 20150   ERROR for  20150 
20151 20152 20153 20155 20157 20158 20159 20160 20161 20162 20164 20166 20167 20168 20170 20174   ERROR fo

In [25]:
# cuento cuantos registros NO puedieron calcularse

tb_final["tn"].is_null().sum()

210

In [26]:
arch_tb_final

'tb_final.csv'

In [27]:
# grabo la historia

tb_final.write_csv(arch_tb_final)

## 1.5 Segunda Pasada AutoARIMA

Me quedaron productos que NO pudo construir el modelo ARIMA debido a no tener suficiente historia para estimar estacionalidad. Reintentamos esos, ahora sin estacionalidad, usando igualmente los demás parámetros de `mejor_params`.

In [28]:
# ahora SIN  estacionalidad

# solo los productos que faltan
productos = tb_final.filter(
    pl.col("tn").is_null()
).select("product_id" ).get_column("product_id").to_list()

# Es fundamental que tb_ventas este ORDENADO
tb_ventas = tb_ventas.sort(["product_id", "periodo"])

for producto in productos:

  print(producto, end=' ')
  tn_values = tb_ventas.filter(pl.col("product_id") == producto).select(["tn"])

  try:
    modelo = auto_arima(
      tn_values,
      seasonal= False,  # Sin estacionalidad
      stepwise=True,
      suppress_warnings=True,
      error_action="ignore",
      max_p=mejor_params.get("max_p", 3), max_q=mejor_params.get("max_q", 3),
      max_P=mejor_params.get("max_P", 2), max_Q=mejor_params.get("max_Q", 2),
      random=True,
      random_state=PARAM['semilla_primigenia'],
      n_fits=mejor_params.get("n_fits", 10)
    )

    forecast = modelo.predict(n_periods=2)
    mesmasdos = forecast[1]

    tb_final = tb_final.with_columns(
       pl.when(pl.col("product_id") == producto)
      .then(mesmasdos if mesmasdos >=0  else 0)
      .otherwise(pl.col("tn")).alias("tn")
    )

  except Exception as e:
    print(f"  ERROR for  {producto} ")

20032 20085 20089 20127 20135 20150 20174 20202 20203 20210 20213 20218 20236 20257 20261 20286 20297 20306 20323 20344 20355 20368 20378 20387 20408 20414 20440 20442 20458 20459 20460 20476 20477 20481 20491 20495 20496 20503 20510 20521 20522 20523 20525 20526 20527 20531 20537 20540 20541 20547 20548 20552 20553 20558 20559 20569 20574 20575 20577 20580 20589 20592 20593 20603 20611 20615 20620 20621 20623 20627 20633 20638 20649 20657 20659 20662 20673 20674 20679 20681 20682 20686 20689 20691 20694 20700 20703 20709 20711 20719 20720 20732 20741 20746 20754 20757 20758 20762 20772 20774 20777 20783 20785 20795 20811 20815 20817 20822 20827 20832 20835 20845 20853 20859 20886 20899 20902 20904 20907 20908 20910 20912 20917 20920 20924 20927 20928 20932 20933 20942 20946 20953 20962 20966 20967 20968 20975 20987 20990 20995 20997 21001 21006 21007 21022 21033 21035 21037 21039 21042 21044 21056 21058 21064 21073 21074 21079 21084 21086 21087 21092 21093 21097 21099 21105 21109 2111

In [29]:
# cuento cuantos registros NO puedieron calcularse aun en este segundo intento
tb_final["tn"].is_null().sum()

0

In [30]:
# vuelvo a grabar
tb_final.write_csv(arch_tb_final)

## 1.6 Submit a Kaggle

In [31]:
os.getcwd()

'/content/.drive/MyDrive/labo3/exp/AA01'

In [32]:
# Submit a Kaggle
if not empiojar:
  archivo= "autoARIMA_bayesiana.csv"
  mensaje= "autoARIMA con parametros optimizados via Optuna, DOS fases"
else:
  archivo= "autoARIMA_bayesiana_empiojado.csv"
  mensaje= "autoARIMA logEMPIOJADO al " + str(empiojar_ruido) + " ,dos fases, params Optuna"

tb_final.write_csv(archivo)

kaggle_submit(PARAM['kaggle_competition'], archivo, mensaje )